In [1]:
# %% [markdown]
# # 09_latency_analysis.ipynb
# Comprehensive latency analysis addressing Reviewer 1 Concern #10:
# "report paired latency differences, median/IQR/p90/p95, disclose
# whether conditions were interleaved or run in separate blocks, and
# account for token/cost."
#
# Covers TWO comparisons using the same methodology:
#   A. v1 (GPT-4o-mini, non-encapsulating) vs v2 (GPT-4o-mini, true
#      info-hiding) -- the architecture comparison
#   B. v2-GPT-4o-mini vs v2-GPT-4o -- the model comparison
#      (Reviewer 2 Concern #2)
#
# Requires: results_phase3_v2.csv (v1), results_phase3_v2_arch.csv
# (v2, GPT-4o-mini), results_phase3_v2_gpt4o.csv (v2, GPT-4o) all
# present in this directory.

# %%
print("SCRIPT VERSION: 2026-07-15-v1")

import pandas as pd
import numpy as np
from scipy import stats

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

df_v1 = pd.read_csv('results_phase3_v2.csv')
df_v2_mini = pd.read_csv('results_phase3_v2_arch.csv')
df_v2_4o = pd.read_csv('results_phase3_v2_gpt4o.csv')

print(f"v1 (GPT-4o-mini):        {len(df_v1)} records")
print(f"v2 GPT-4o-mini:          {len(df_v2_mini)} records")
print(f"v2 GPT-4o:               {len(df_v2_4o)} records")

# %% [markdown]
# ## 0. Interleaving disclosure (per R1's explicit request)
#
# This is a methods-section fact, not something computed from the
# data -- documented here so it's derived from the actual experiment
# code structure, not asserted without basis.

# %%
print("=" * 70)
print("INTERLEAVING DISCLOSURE")
print("=" * 70)
print("""
v1 and v2-GPT-4o-mini were NOT interleaved: v1 was generated by a
separate notebook (credit_rating_benchmark_experiment.ipynb, the
original v1 implementation) run in an entirely separate session from
v2's 04_full_experiment.py. The two runs do not share wall-clock
time, so any latency difference between v1 and v2 could in principle
reflect API-side conditions specific to when each was run (server
load, routing, etc.) rather than a difference intrinsic to the two
prompt architectures. This is a genuine limitation of the latency
comparison between v1 and v2 and is reported as such below rather
than assuming it away.

v2-GPT-4o-mini and v2-GPT-4o were ALSO run as separate blocks (in
separate notebook runs -- 04_full_experiment.py and
07_multi_model_experiment.py respectively), not interleaved within a
single loop. The same caveat applies to the model comparison.

Within EACH individual run, however, the 5 iterations for a given
question ARE processed as a tight sequential loop (see the "for
iteration in range(N_ITER)" structure in both 04_full_experiment.py
and 07_multi_model_experiment.py), so within-run timing is at least
locally comparable across questions in that run.
""")

# %% [markdown]
# ## 1. Clean latency data (drop API-failure rows where latency_ms
#     is None/NaN -- these reflect exception handling time, not real
#     model response latency, and would distort the distribution)

# %%
def clean_latency(df, label):
    n_total = len(df)
    n_valid = df['latency_ms'].notna().sum()
    n_dropped = n_total - n_valid
    print(f"{label}: {n_valid}/{n_total} records have valid latency_ms "
          f"({n_dropped} dropped -- API call failures or compile-only records)")
    return df[df['latency_ms'].notna()].copy()

df_v1_clean = clean_latency(df_v1, "v1")
df_v2_mini_clean = clean_latency(df_v2_mini, "v2 GPT-4o-mini")
df_v2_4o_clean = clean_latency(df_v2_4o, "v2 GPT-4o")

# %% [markdown]
# ## 2. Full distributional summary (mean, median, IQR, p90, p95, max)
#     per R1's explicit request for more than just mean/median

# %%
def latency_distribution(df, label):
    s = df['latency_ms']
    return pd.Series({
        'n': len(s),
        'mean': s.mean(),
        'median': s.median(),
        'std': s.std(),
        'p25': s.quantile(0.25),
        'p75': s.quantile(0.75),
        'iqr': s.quantile(0.75) - s.quantile(0.25),
        'p90': s.quantile(0.90),
        'p95': s.quantile(0.95),
        'max': s.max(),
    }, name=label)

print("=" * 70)
print("FULL LATENCY DISTRIBUTION (ms)")
print("=" * 70)
dist_table = pd.concat([
    latency_distribution(df_v1_clean, 'v1 (GPT-4o-mini)'),
    latency_distribution(df_v2_mini_clean, 'v2 (GPT-4o-mini)'),
    latency_distribution(df_v2_4o_clean, 'v2 (GPT-4o)'),
], axis=1)
print(dist_table)

# %% [markdown]
# ## 3. Comparison A: v1 vs v2 (GPT-4o-mini), paired at question level
#
# Paired at the QUESTION level (mean latency per question), matching
# the same aggregation principle used throughout this revision's
# accuracy analysis (06_statistical_reanalysis.py), for consistency
# and to avoid the independence-violation issue R1 raised for
# accuracy (Concern #2) recurring here for latency.

# %%
def question_level_latency(df):
    return df.groupby('question_id')['latency_ms'].mean()

q_v1 = question_level_latency(df_v1_clean)
q_v2_mini = question_level_latency(df_v2_mini_clean)
q_v2_4o = question_level_latency(df_v2_4o_clean)

print("=" * 70)
print("COMPARISON A: v1 vs v2 (GPT-4o-mini), question-level paired")
print("=" * 70)

common_a = sorted(set(q_v1.index) & set(q_v2_mini.index))
print(f"n = {len(common_a)} question pairs")

v1_aligned_a = q_v1.loc[common_a].values
v2_aligned_a = q_v2_mini.loc[common_a].values
diff_a = v2_aligned_a - v1_aligned_a

print(f"v1 mean question-level latency:  {v1_aligned_a.mean():,.1f} ms")
print(f"v2 mean question-level latency:  {v2_aligned_a.mean():,.1f} ms")
print(f"Mean paired difference (v2 - v1): {diff_a.mean():+,.1f} ms")
print(f"Median paired difference:         {np.median(diff_a):+,.1f} ms")

t_stat_a, t_p_a = stats.ttest_rel(v2_aligned_a, v1_aligned_a)
w_stat_a, w_p_a = stats.wilcoxon(v2_aligned_a, v1_aligned_a)
print(f"\nPaired t-test:        t={t_stat_a:.4f}, p={t_p_a:.4g}")
print(f"Wilcoxon signed-rank: W={w_stat_a:.4f}, p={w_p_a:.4g}")
print(f"\nQuestions where v2 slower: {(diff_a > 0).sum()}, "
      f"v1 slower: {(diff_a < 0).sum()}, tied: {(diff_a == 0).sum()}")

# %% [markdown]
# ## 4. Comparison B: GPT-4o-mini vs GPT-4o (both v2), paired at
#     question level

# %%
print("\n" + "=" * 70)
print("COMPARISON B: v2 GPT-4o-mini vs v2 GPT-4o, question-level paired")
print("=" * 70)

common_b = sorted(set(q_v2_mini.index) & set(q_v2_4o.index))
print(f"n = {len(common_b)} question pairs")

mini_aligned_b = q_v2_mini.loc[common_b].values
o4_aligned_b = q_v2_4o.loc[common_b].values
diff_b = o4_aligned_b - mini_aligned_b

print(f"GPT-4o-mini mean question-level latency: {mini_aligned_b.mean():,.1f} ms")
print(f"GPT-4o mean question-level latency:      {o4_aligned_b.mean():,.1f} ms")
print(f"Mean paired difference (4o - mini): {diff_b.mean():+,.1f} ms")
print(f"Median paired difference:           {np.median(diff_b):+,.1f} ms")

t_stat_b, t_p_b = stats.ttest_rel(o4_aligned_b, mini_aligned_b)
w_stat_b, w_p_b = stats.wilcoxon(o4_aligned_b, mini_aligned_b)
print(f"\nPaired t-test:        t={t_stat_b:.4f}, p={t_p_b:.4g}")
print(f"Wilcoxon signed-rank: W={w_stat_b:.4f}, p={w_p_b:.4g}")
print(f"\nQuestions where GPT-4o slower: {(diff_b > 0).sum()}, "
      f"GPT-4o-mini slower: {(diff_b < 0).sum()}, tied: {(diff_b == 0).sum()}")

# %% [markdown]
# ## 5. Question-level bootstrap 95% CI for both comparisons

# %%
def question_level_bootstrap(values_a, values_b, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(values_a)
    boot_diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        boot_diffs[i] = values_b[idx].mean() - values_a[idx].mean()
    return np.percentile(boot_diffs, [2.5, 97.5])

ci_a = question_level_bootstrap(v1_aligned_a, v2_aligned_a)
ci_b = question_level_bootstrap(mini_aligned_b, o4_aligned_b)

print("\n" + "=" * 70)
print("BOOTSTRAP 95% CI (question-level, n_boot=5000)")
print("=" * 70)
print(f"Comparison A (v2 - v1) latency diff:              "
      f"[{ci_a[0]:+,.1f}, {ci_a[1]:+,.1f}] ms")
print(f"Comparison B (GPT-4o - GPT-4o-mini) latency diff: "
      f"[{ci_b[0]:+,.1f}, {ci_b[1]:+,.1f}] ms")

# %% [markdown]
# ## 6. Prompt length vs latency (supports/refutes a length-driven
#     explanation for any latency difference)

# %%
print("\n" + "=" * 70)
print("PROMPT LENGTH vs LATENCY")
print("=" * 70)

for label, df in [("v1", df_v1_clean), ("v2 GPT-4o-mini", df_v2_mini_clean),
                   ("v2 GPT-4o", df_v2_4o_clean)]:
    if 'prompt_chars' in df.columns and df['prompt_chars'].notna().sum() > 2:
        sub = df[['prompt_chars', 'latency_ms']].dropna()
        r, p = stats.pearsonr(sub['prompt_chars'], sub['latency_ms'])
        print(f"{label:<18} mean prompt_chars={sub['prompt_chars'].mean():>8,.0f}  "
              f"corr(prompt_chars, latency)={r:+.3f} (p={p:.4g}, n={len(sub)})")
    else:
        print(f"{label:<18} prompt_chars not available or insufficient data")

# %% [markdown]
# ## 7. Token/cost estimation
#
# Neither utils.py's query functions nor the OpenAI client call here
# capture actual token usage from the API response (the `usage`
# field), so token counts below are ESTIMATED from character counts
# using a standard approximation (~4 characters per token for
# English text), not measured directly. This is disclosed explicitly
# rather than presented as precise -- if exact token/cost figures are
# needed for the manuscript, utils.py's query functions should be
# extended to capture `res.usage` from the API response and this
# section re-run against a fresh experiment (see note at the end).

# %%
print("\n" + "=" * 70)
print("TOKEN/COST ESTIMATION (approximate -- see note above)")
print("=" * 70)

CHARS_PER_TOKEN = 4  # standard rough approximation for English text

# Approximate per-1M-token pricing (illustrative; verify current
# pricing before using these numbers in the manuscript, as API
# pricing changes over time and is not something this analysis can
# verify on its own)
PRICING_PER_1M_INPUT_TOKENS = {'gpt-4o-mini': 0.15, 'gpt-4o': 2.50}
PRICING_PER_1M_OUTPUT_TOKENS = {'gpt-4o-mini': 0.60, 'gpt-4o': 10.00}
OUTPUT_TOKEN_ESTIMATE = 100  # concept-SQL responses are short; rough estimate

def estimate_cost(df, model_key, label):
    if 'prompt_chars' not in df.columns or df['prompt_chars'].isna().all():
        print(f"{label}: prompt_chars not available, cannot estimate")
        return
    mean_chars = df['prompt_chars'].mean()
    mean_input_tokens = mean_chars / CHARS_PER_TOKEN
    input_cost_per_call = (mean_input_tokens / 1_000_000) * PRICING_PER_1M_INPUT_TOKENS[model_key]
    output_cost_per_call = (OUTPUT_TOKEN_ESTIMATE / 1_000_000) * PRICING_PER_1M_OUTPUT_TOKENS[model_key]
    total_cost_per_call = input_cost_per_call + output_cost_per_call
    n_calls = len(df)
    print(f"{label}:")
    print(f"  Mean prompt length: {mean_chars:,.0f} chars (~{mean_input_tokens:,.0f} tokens, estimated)")
    print(f"  Estimated cost per call: ${total_cost_per_call:.5f}")
    print(f"  Estimated total cost ({n_calls} calls): ${total_cost_per_call * n_calls:.2f}")

estimate_cost(df_v2_mini_clean, 'gpt-4o-mini', 'v2 GPT-4o-mini')
estimate_cost(df_v2_4o_clean, 'gpt-4o', 'v2 GPT-4o')

print("\n!! These are ROUGH ESTIMATES based on character-count approximation,")
print("   not measured token usage. Verify current API pricing before citing")
print("   in the manuscript, and consider extending query_semantic_layer_v2()")
print("   to capture res.usage.prompt_tokens / res.usage.completion_tokens")
print("   directly for exact figures in a future run.")

# %% [markdown]
# ## 8. Final summary for the manuscript

# %%
print("\n" + "=" * 70)
print("FINAL LATENCY SUMMARY (for manuscript Section 4.3 / 5.5 revision)")
print("=" * 70)
print(f"""
  Comparison A: v1 vs v2 (GPT-4o-mini)
    Mean latency:     v1={v1_aligned_a.mean():,.0f}ms  v2={v2_aligned_a.mean():,.0f}ms
    Paired diff:       {diff_a.mean():+,.0f}ms (median {np.median(diff_a):+,.0f}ms)
    Paired t-test:      t={t_stat_a:.2f}, p={t_p_a:.4g}
    Bootstrap 95% CI:   [{ci_a[0]:+,.0f}, {ci_a[1]:+,.0f}] ms
    NOT interleaved -- run in separate sessions (see Section 0)

  Comparison B: GPT-4o-mini vs GPT-4o (both v2)
    Mean latency:     mini={mini_aligned_b.mean():,.0f}ms  4o={o4_aligned_b.mean():,.0f}ms
    Paired diff:       {diff_b.mean():+,.0f}ms (median {np.median(diff_b):+,.0f}ms)
    Paired t-test:      t={t_stat_b:.2f}, p={t_p_b:.4g}
    Bootstrap 95% CI:   [{ci_b[0]:+,.0f}, {ci_b[1]:+,.0f}] ms
    NOT interleaved -- run in separate sessions (see Section 0)

  Tail latency (p90/p95) is reported in the distribution table above
  (Section 2) for all three conditions -- use those columns directly
  for the manuscript rather than mean/median alone, per R1's request.

  Token/cost figures (Section 7) are ESTIMATES from character counts,
  not measured API usage -- flag this limitation if citing them.
""")

print("Done. This addresses Reviewer 1 Concern #10's four specific")
print("requirements: (1) paired differences -- Sections 3-4; (2) median/")
print("IQR/p90/p95 -- Section 2; (3) interleaving disclosure -- Section 0;")
print("(4) token/cost -- Section 7 (with explicit estimation caveat).")

SCRIPT VERSION: 2026-07-15-v1
v1 (GPT-4o-mini):        600 records
v2 GPT-4o-mini:          300 records
v2 GPT-4o:               300 records
INTERLEAVING DISCLOSURE

v1 and v2-GPT-4o-mini were NOT interleaved: v1 was generated by a
separate notebook (credit_rating_benchmark_experiment.ipynb, the
original v1 implementation) run in an entirely separate session from
v2's 04_full_experiment.py. The two runs do not share wall-clock
time, so any latency difference between v1 and v2 could in principle
reflect API-side conditions specific to when each was run (server
load, routing, etc.) rather than a difference intrinsic to the two
prompt architectures. This is a genuine limitation of the latency
comparison between v1 and v2 and is reported as such below rather
than assuming it away.

v2-GPT-4o-mini and v2-GPT-4o were ALSO run as separate blocks (in
separate notebook runs -- 04_full_experiment.py and
07_multi_model_experiment.py respectively), not interleaved within a
single loop. The same ca